# task_rules_ver05_trial_08

このノートブックは `annotations_gt_task_ver10.json` から `tasks[0].action == apply_style` の先頭エントリを探し、`apply_style` を単体実行して確認するための検証用です。

初期フレームに Lora処理をする

In [ ]:
from pathlib import Path
import json
import sys
import logging

WORKSPACE = Path('/workspace')
VIDEO_DIR = WORKSPACE / 'data' / 'videos'
RULES_PATH = WORKSPACE / 'logs' / 'submit' / 'submission_ver05_json' / 'task_rules_ver05.json'
TASK_DECOMP_PATH = WORKSPACE / 'logs' / 'submit' / 'submission_ver05_json' / 'task_decomposition_ver05.json'
ANNOT_PATH = WORKSPACE / 'data' / 'annotations_gt_task_ver10.json'
OUT_DIR = WORKSPACE / 'logs' / 'notebook' / 'task_rules_ver05_trial_08'
OUT_DIR.mkdir(parents=True, exist_ok=True)

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from src.postprocess import task_rules_ver05_functions as task_rule_funcs
from src.utils.video_utility import load_video, write_video, show_before_after

rules_payload = json.loads(RULES_PATH.read_text(encoding='utf-8'))
rules = rules_payload['actions']
task_rows = json.loads(TASK_DECOMP_PATH.read_text(encoding='utf-8'))
instruction_by_video = {str(r.get('video_path', '')): str(r.get('instruction', '')) for r in task_rows}

logger = logging.getLogger('task_rules_ver05_trial_08')
logger.setLevel(logging.INFO)
if not logger.handlers:
    h = logging.StreamHandler()
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter('[%(levelname)s] %(message)s'))
    logger.addHandler(h)

def run_action_core(video_path_in, video_path_out, action, params_override=None, instruction_override=None, show_compare=False):
    frames, fps, width, height = load_video(video_path_in)
    rule = dict(rules.get(action, {}))
    method = str(rule.get('method', 'identity'))

    params = dict(rule.get('params', {}))
    if params_override:
        params.update(params_override)

    if instruction_override is None:
        instruction = instruction_by_video.get(Path(video_path_in).name, '')
    else:
        instruction = instruction_override

    out_frames = task_rule_funcs.run_method(
        method=method,
        frames=frames,
        params=params,
        instruction=instruction,
        logger=logger,
    )
    write_video(video_path_out, out_frames, fps, width, height)
    if show_compare:
        show_before_after(video_path_in, video_path_out)
    return instruction, method, params

print('rules :', len(rules))
print('RULES_PATH:', RULES_PATH)
print('TASK_DECOMP_PATH:', TASK_DECOMP_PATH)
print('ANNOT_PATH:', ANNOT_PATH)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


rules : 44
RULES_PATH: /workspace/logs/submit/submission_ver05_json/task_rules_ver05.json
TASK_DECOMP_PATH: /workspace/logs/submit/submission_ver05_json/task_decomposition_ver05.json
ANNOT_PATH: /workspace/data/annotations_gt_task_ver10.json


In [2]:
# tasks[0].action == apply_style の先頭エントリを準備
SHOW_COMPARE = True
STYLE = 'oil_painting'  #'ghibli'
ann_rows = json.loads(ANNOT_PATH.read_text(encoding='utf-8'))
if not ann_rows:
    raise ValueError('annotations is empty')

ANNOTATION_ID = None
entry = None
for i, row in enumerate(ann_rows):
    tasks = row.get('tasks', [])
    if tasks and str(tasks[0].get('action', '')) == 'apply_style':
        ANNOTATION_ID = i
        entry = dict(row)
        break

if entry is None:
    raise ValueError('tasks[0].action == apply_style のエントリが見つかりません')

TARGET_VIDEO = str(entry.get('video_path', ''))
if not TARGET_VIDEO:
    raise ValueError('selected annotation has empty video_path')

first_task = dict(entry.get('tasks', [])[0])
instruction = str(entry.get('instruction', ''))
video_in = VIDEO_DIR / TARGET_VIDEO
if not video_in.exists():
    raise FileNotFoundError(f'input video not found: {video_in}')

SEQ_OUT_DIR = OUT_DIR / f'apply_style_top{ANNOTATION_ID+1}' / Path(TARGET_VIDEO).stem
SEQ_OUT_DIR.mkdir(parents=True, exist_ok=True)

print('annotation id  :', ANNOTATION_ID)
print('target video   :', TARGET_VIDEO)
print('class/subclass :', entry.get('class'), '/', entry.get('subclass'))
print('instruction    :', instruction)
print('gt task action :', first_task.get('action', ''))
print('apply style    :', STYLE)

annotation id  : 5
target video   : 94msufYZzaQ_26_0to273.mp4
class/subclass : Style Editing / Ukiyo-e
instruction    : Transform the entire video into a traditional Japanese Ukiyo-e woodblock print style. Ensure the identities and facial expressions of the two men are preserved while applying the bold outlines and flat color palettes characteristic of the style. Maintain the recognizable layout of the workshop background with consistent textures and no temporal flickering between frames.
gt task action : apply_style
apply style    : oil_painting


In [ ]:
# apply_style を1つだけ実行
import importlib
importlib.reload(task_rule_funcs)

action = 'apply_style'
params = dict(first_task.get('params', {}))
params['_action'] = action
params['_constraints'] = list(first_task.get('constraints', []))
params['action'] = action
params['constraints'] = list(first_task.get('constraints', []))
params['style'] = STYLE
params['video_path'] = TARGET_VIDEO
if first_task.get('target', None) is not None:
    params['target'] = first_task.get('target', '')

out_path = SEQ_OUT_DIR / f"{Path(TARGET_VIDEO).stem}__task01_{action}_{STYLE}.mp4"
_instruction, method, used_params = run_action_core(
    video_in,
    out_path,
    action,
    params_override=params,
    instruction_override=instruction,
    show_compare=SHOW_COMPARE,
)

print('video      :', video_in.name)
print('action     :', action)
print('style      :', STYLE)
print('method     :', method)
print('out        :', out_path)
print('used params:', used_params)

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 6537.81it/s]]
CLIPTextModel LOAD REPORT from: /workspace/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 396/396 [00:00<00:00, 4743.58it/s]7.40it/s]
StableDiffusionSafetyChecker LOAD REPORT from: /workspace/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Note

xformers not available, skip


100%|██████████| 3/3 [00:03<00:00,  1.32s/it]


video      : 94msufYZzaQ_26_0to273.mp4
action     : apply_style
style      : oil_painting
method     : stylize
out        : /workspace/logs/notebook/task_rules_ver05_trial_08/apply_style_top6/94msufYZzaQ_26_0to273/94msufYZzaQ_26_0to273__task01_apply_style_oil_painting.mp4
used params: {'style': 'oil_painting', '_action': 'apply_style', '_constraints': [], 'action': 'apply_style', 'constraints': [], 'video_path': '94msufYZzaQ_26_0to273.mp4', 'target': 'full_frame', 'instruction': 'Transform the entire video into a traditional Japanese Ukiyo-e woodblock print style. Ensure the identities and facial expressions of the two men are preserved while applying the bold outlines and flat color palettes characteristic of the style. Maintain the recognizable layout of the workshop background with consistent textures and no temporal flickering between frames.'}


: 